# <font color="#418FDE" size="6.5" uppercase>**Unüberwacht lernen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Vergleichen KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture. 
- Wenden IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierte Daten an. 
- Untersuchen semi-supervised Verfahren mit wenigen Labels und dokumentieren Unsicherheit. 


## **1. Clusterverfahren vergleichen**

### **1.1. KMeans im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_01.jpg?v=1787649335" width="250">



>* KMeans gruppiert Daten um erklärbare Zentren
>* Funktioniert schlecht bei komplexen Clusterformen

>* Schnell, aber Clusterzahl bewusst wählen
>* Skalierung, Ausreißer und Fachkontext prüfen

>* KMeans ordnet Punkte eindeutig einem Cluster zu
>* Andere Verfahren modellieren Unsicherheit und Formen flexibler



In [ ]:
#@title Python-Code - KMeans im Vergleich

# Wir vergleichen KMeans mit einer flexibleren Clusterform.
# Skalierung macht Distanzvergleiche fairer und stabiler.
# Die Grafik zeigt typische Stärken und Grenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# Ein zweidimensionaler Datensatz zeigt nicht runde Cluster.
features, true_labels = make_moons(
    n_samples=300,
    noise=0.07,
    random_state=42,
)

# KMeans arbeitet mit Abständen, daher skalieren wir zuerst.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if scaled_features.shape != (300, 2):
    raise ValueError("Die Beispieldaten sollten 300 Zeilen und 2 Spalten haben.")

# KMeans sucht zwei kompakte Gruppen mit Zentroiden.
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
predicted_labels = kmeans.fit_predict(scaled_features)

# Der ARI vergleicht gefundene Cluster mit bekannten Beispielgruppen.
ari_score = adjusted_rand_score(true_labels, predicted_labels)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Adjusted Rand Index für KMeans: {ari_score:.2f}")
print("Merke: KMeans bevorzugt kompakte, rundliche Cluster.")

# Die Zentroiden zeigen die harte Zuordnung von KMeans.
centers = kmeans.cluster_centers_
fig, ax = plt.subplots(figsize=(7, 5))

# Farben zeigen die von KMeans gefundenen Gruppen.
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=predicted_labels,
    cmap="viridis",
    s=35,
    alpha=0.85,
)

# Schwarze Kreuze markieren die berechneten Zentroiden.
ax.scatter(
    centers[:, 0],
    centers[:, 1],
    c="black",
    marker="X",
    s=180,
    label="Zentroide",
)

# Achsen und Titel erklären die sichtbare Entscheidung.
ax.set_title("KMeans auf nicht rundlichen Clustern")
ax.set_xlabel("skaliertes Merkmal 1")
ax.set_ylabel("skaliertes Merkmal 2")
ax.legend()
plt.show()



### **1.2. MiniBatch und Silhouette**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_02.jpg?v=1787649337" width="250">



>* MiniBatchKMeans nutzt kleine Stichproben statt aller Daten
>* Schneller, aber manchmal etwas weniger präzise

>* Silhouette bewertet Zusammenhalt und Trennung von Clustern
>* Sie hilft, passende Clusteranzahlen zu vergleichen

>* Silhouette-Werte immer fachlich kritisch prüfen
>* MiniBatchKMeans mit weiteren Prüfungen bewerten



In [ ]:
#@title Python-Code - MiniBatch und Silhouette

# Wir vergleichen MiniBatchKMeans mit Silhouette-Werten.
# Die Clusteranzahl beeinflusst die Bewertung deutlich.
# Die Grafik zeigt die beste einfache Wahl.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine, gut sichtbare Beispieldaten.
features, true_groups = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=1.2,
    random_state=42,
)

# Skalierung macht Abstände zwischen Merkmalen vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Diese Prüfung schützt vor unerwarteten Datenformen.
if scaled_features.shape != (600, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

cluster_counts = [2, 3, 4, 5, 6]
silhouette_scores = []

# Wir testen mehrere mögliche Clusteranzahlen.
for cluster_count in cluster_counts:
    model = MiniBatchKMeans(
        n_clusters=cluster_count,
        batch_size=64,
        n_init=10,
        random_state=42,
    )
    labels = model.fit_predict(scaled_features)
    score = silhouette_score(scaled_features, labels)
    silhouette_scores.append(score)

best_index = int(np.argmax(silhouette_scores))
best_count = cluster_counts[best_index]
best_score = silhouette_scores[best_index]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Beste getestete Clusteranzahl: {best_count}")
print(f"Bester Silhouette-Wert: {best_score:.3f}")

# Die Linie macht den Vergleich der Werte sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_counts, silhouette_scores, marker="o", color="tab:blue")
ax.scatter(best_count, best_score, s=120, color="tab:orange", label="bester Wert")

ax.set_title("MiniBatchKMeans: Silhouette nach Clusteranzahl")
ax.set_xlabel("Anzahl der Cluster")
ax.set_ylabel("Silhouette-Wert")
ax.set_xticks(cluster_counts)
ax.set_ylim(0, 1)
ax.legend()
plt.show()



### **1.3. Hierarchisches Clustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_03.jpg?v=1787649339" width="250">



>* Keine feste Clusterzahl nötig
>* Baum zeigt feine und grobe Ähnlichkeiten

>* Verknüpfung bestimmt Clusterform und Interpretation
>* Interpretierbar, aber rechenintensiv bei großen Daten

>* Schnitthöhe steuert Anzahl und Feinheit der Cluster
>* Transparent, aber ohne Rauschen oder Wahrscheinlichkeiten



In [ ]:
#@title Python-Code - Hierarchisches Clustering

# Dieses Beispiel zeigt hierarchisches Clustering mit kleinen Punktgruppen.
# Die Baumstruktur macht schrittweise Zusammenführungen sichtbar.
# Der Plot zeigt Cluster bei gewählter Schnitthöhe.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn import __version__ as sklearn_version

# Wir erzeugen übersichtliche Daten mit drei natürlichen Gruppen.
features, true_groups = make_blobs(
    n_samples=24,
    centers=3,
    cluster_std=0.65,
    random_state=42,
)

# Eine einfache Prüfung verhindert missverständliche Formen.
if features.shape != (24, 2):
    raise ValueError("Die Beispieldaten sollten 24 Zeilen und 2 Spalten haben.")

# Agglomeratives Clustering baut die Hierarchie von unten nach oben.
model = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward",
    compute_distances=True,
)

# Das Modell wird einmal auf die Punktdaten angepasst.
cluster_labels = model.fit_predict(features)

# Für das Dendrogramm bauen wir die SciPy-Linkage-Matrix nach.
counts = np.zeros(model.children_.shape[0])
n_samples = len(model.labels_)

# Jede Zeile beschreibt, wie viele Punkte zusammengeführt wurden.
for index, merge in enumerate(model.children_):
    current_count = 0
    for child_index in merge:
        if child_index < n_samples:
            current_count += 1
        else:
            current_count += counts[child_index - n_samples]
    counts[index] = current_count

# Die Linkage-Matrix enthält Paare, Distanzen und Gruppengrößen.
linkage_matrix = np.column_stack(
    [model.children_, model.distances_, counts]
).astype(float)

# Kurze Ausgaben verbinden Modellidee und Ergebnis.
print(f"scikit-learn Version: {sklearn_version}")
print(f"Beobachtungen: {n_samples}, Merkmale: {features.shape[1]}")
print(f"Gewählte Clusterzahl durch Schnitt: {len(np.unique(cluster_labels))}")

# Das Dendrogramm zeigt frühe und späte Zusammenführungen.
fig, ax = plt.subplots(figsize=(9, 4))
dendrogram(linkage_matrix, ax=ax, color_threshold=5.0)
ax.axhline(y=5.0, color="black", linestyle="--", label="Schnitthöhe")

# Achsen und Titel machen die Baumstruktur lesbar.
ax.set_title("Hierarchisches Clustering als Dendrogramm")
ax.set_xlabel("Beobachtung oder zusammengeführtes Cluster")
ax.set_ylabel("Zusammenführungsdistanz")
ax.legend()
plt.show()



## **2. Dichte und Anomalien**

### **2.1. Dichtebasierte Cluster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_01.jpg?v=1787649324" width="250">



>* Cluster entstehen in dichten Datenbereichen
>* Anomalien liegen oft in dünnen Regionen

>* Skalierung verhindert verzerrte Dichte durch große Wertebereiche
>* Standardisierung macht Abstände und Muster vergleichbar

>* Dünne Regionen zeigen mögliche Ausreißer
>* Fachwissen prüft Bedeutung und Risiko



In [ ]:
#@title Python-Code - Dichtebasierte Cluster

# Wir untersuchen dichtebasierte Cluster mit DBSCAN.
# Skalierung macht Abstände zwischen Merkmalen vergleichbarer.
# Randpunkte erscheinen als mögliche Anomalien.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

# Wir erzeugen zwei gekrümmte Punktwolken mit Zusatzrauschen.
features, true_labels = make_moons(
    n_samples=260,
    noise=0.07,
    random_state=42,
)

# Einige entfernte Punkte simulieren ungewöhnliche Beobachtungen.
outliers = np.array([
    [-1.4, 1.2],
    [2.2, -0.7],
    [2.5, 0.9],
    [-1.2, -0.6],
])

# Wir hängen die Ausreißer an die normalen Daten an.
features = np.vstack([features, outliers])

# Eine einfache Prüfung verhindert unerwartete Formfehler.
if features.shape[1] != 2:
    raise ValueError("Dieses Beispiel erwartet genau zwei Merkmale.")

# StandardScaler passt die Skalen vor der Abstandsmessung an.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# DBSCAN sucht dichte Regionen und markiert Rauschen mit minus eins.
model = DBSCAN(eps=0.28, min_samples=6)
cluster_labels = model.fit_predict(scaled_features)

# Wir zählen Cluster ohne die Rauschmarkierung minus eins.
cluster_ids = set(cluster_labels)
cluster_count = len(cluster_ids - {-1})
noise_count = int(np.sum(cluster_labels == -1))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Gefundene dichte Cluster: {cluster_count}")
print(f"Als Rauschen markierte Punkte: {noise_count}")

# Farben zeigen Cluster, schwarze Kreuze zeigen dünn besiedelte Punkte.
fig, ax = plt.subplots(figsize=(7, 5))
normal_mask = cluster_labels != -1
noise_mask = cluster_labels == -1

scatter = ax.scatter(
    features[normal_mask, 0],
    features[normal_mask, 1],
    c=cluster_labels[normal_mask],
    cmap="viridis",
    s=35,
    label="Dichte Cluster",
)

ax.scatter(
    features[noise_mask, 0],
    features[noise_mask, 1],
    c="black",
    marker="x",
    s=80,
    label="Rauschen oder Anomalie",
)

ax.set_title("DBSCAN: dichte Cluster und mögliche Anomalien")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **2.2. Gaußsche Mischmodelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_02.jpg?v=1787649328" width="250">



>* Cluster als Wahrscheinlichkeiten statt fester Grenzen
>* Dichte zeigt passende und auffällige Punkte

>* Skalierung verhindert verzerrte Verteilungsmodelle
>* Niedrige Wahrscheinlichkeiten weisen auf Anomalien hin

>* Modellannahmen und Komponentenzahl kritisch prüfen
>* Unsicherheit mit Fachwissen und Vergleichen einordnen



In [ ]:
#@title Python-Code - Gaußsche Mischmodelle

# Wir modellieren Dichte mit gaußschen Mischmodellen.
# Skalierung macht Merkmale für das Modell vergleichbar.
# Niedrige Dichtewerte markieren mögliche Anomalien.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Wir erzeugen normale Gruppen und wenige Ausreißer.
normal_data, true_groups = make_blobs(
    n_samples=240,
    centers=[[-2, 0], [2, 1]],
    cluster_std=[0.7, 0.6],
    random_state=42,
)

outliers = np.array([[-5.0, 4.0], [5.0, -3.0], [0.0, 5.0], [4.5, 4.0]])
raw_data = np.vstack([normal_data, outliers])

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if raw_data.shape != (244, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Die Skalierung wird vor dem Mischmodell gelernt.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(raw_data)

# Das Modell lernt zwei glockenförmige Komponenten.
gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=42)
gmm.fit(scaled_data)

# Niedrige Log-Wahrscheinlichkeit bedeutet geringe Modellpassung.
log_density = gmm.score_samples(scaled_data)
threshold = np.percentile(log_density, 5)
is_anomaly = log_density < threshold

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Gelernte Komponenten: {gmm.n_components}")
print(f"Anomalie-Schwelle: {threshold:.2f}")
print(f"Markierte Anomalien: {int(is_anomaly.sum())} von {len(raw_data)}")

# Die Grafik zeigt Dichteentscheidung und markierte Punkte.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(raw_data[~is_anomaly, 0], raw_data[~is_anomaly, 1], s=35, label="typisch")
ax.scatter(raw_data[is_anomaly, 0], raw_data[is_anomaly, 1], s=80, label="auffällig")
ax.set_title("Gaußsches Mischmodell: niedrige Dichte als Anomalie")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **2.3. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_03.jpg?v=1787649326" width="250">



>* Anomalien weichen von typischen Datenmustern ab
>* Sie sind Hinweise, nicht automatisch Fehler

>* Drei Verfahren erkennen Anomalien unterschiedlich.
>* Skalierung verhindert verzerrte Entscheidungen.

>* Anomalien sind Hinweise, keine endgültigen Urteile
>* Dokumentation, Fairnessprüfung und Kontext bleiben wichtig



In [ ]:
#@title Python-Code - Anomalien erkennen

# Wir erkennen Anomalien in skalierten zweidimensionalen Daten.
# IsolationForest markiert ungewöhnliche Punkte ohne gelabelte Beispiele.
# Die Grafik zeigt normale und auffällige Beobachtungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kompakte normale Punkte und wenige Ausreißer.
normal_points, _ = make_blobs(
    n_samples=220,
    centers=[(-2, -1), (2, 1)],
    cluster_std=0.65,
    random_state=42,
)

rng = np.random.default_rng(42)
outlier_points = rng.uniform(low=-6, high=6, size=(18, 2))
raw_data = np.vstack([normal_points, outlier_points])

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if raw_data.shape != (238, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Skalierung verhindert, dass ein Merkmal zu stark dominiert.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(raw_data)

# IsolationForest isoliert ungewöhnliche Punkte besonders schnell.
model = IsolationForest(contamination=0.08, random_state=42)
predictions = model.fit_predict(scaled_data)
anomaly_scores = -model.decision_function(scaled_data)

# Wir zählen, wie viele Punkte als auffällig gelten.
anomaly_mask = predictions == -1
anomaly_count = int(np.sum(anomaly_mask))
mean_score = float(np.mean(anomaly_scores[anomaly_mask]))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Beobachtungen insgesamt: {len(raw_data)}")
print(f"Als Anomalien markiert: {anomaly_count}")
print(f"Mittlerer Anomaliewert dieser Punkte: {mean_score:.3f}")

# Die Grafik zeigt Entscheidungen im skalierten Merkmalsraum.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    scaled_data[~anomaly_mask, 0],
    scaled_data[~anomaly_mask, 1],
    s=28,
    label="unauffällig",
    alpha=0.75,
)

ax.scatter(
    scaled_data[anomaly_mask, 0],
    scaled_data[anomaly_mask, 1],
    s=70,
    label="Anomalie",
    marker="x",
)

ax.set_title("Anomalien mit IsolationForest nach Skalierung")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend()
plt.show()



## **3. Lernen mit wenigen Labels**

### **3.1. Labels weitergeben**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_01.jpg?v=1787649334" width="250">



>* Labels über Ähnlichkeiten auf unbeschriftete Daten übertragen
>* Hilft bei wenigen, teuren oder fehlenden Labels

>* Gute Textrepräsentationen machen Ähnlichkeit nutzbar
>* Klare Gruppen erleichtern vorsichtige Label-Weitergabe

>* Weitergegebene Labels bleiben vorläufige Annahmen
>* Herkunft, Sicherheit und Fehler transparent dokumentieren



In [ ]:
#@title Python-Code - Labels weitergeben

# Wir geben wenige Labels vorsichtig weiter.
# LabelSpreading nutzt Nachbarschaften zwischen ähnlichen Punkten.
# Unsichere Vorhersagen werden sichtbar dokumentiert.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.semi_supervised import LabelSpreading

# Wir erzeugen drei gut sichtbare Gruppen.
features, true_labels = make_blobs(
    n_samples=90,
    centers=3,
    cluster_std=1.15,
    random_state=42,
)

# Nur wenige Punkte behalten ihr echtes Label.
known_indices = np.array([0, 1, 2, 30, 31, 32, 60, 61, 62])
partial_labels = np.full(true_labels.shape, -1)
partial_labels[known_indices] = true_labels[known_indices]

# Diese Prüfung schützt vor einem leeren Startsignal.
if np.sum(partial_labels != -1) == 0:
    raise ValueError("Mindestens ein bekanntes Label wird benötigt.")

# Das Modell verteilt Labels über ähnliche Nachbarn.
model = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=100)
model.fit(features, partial_labels)

# Wahrscheinlichkeiten zeigen die Unsicherheit der Weitergabe.
predicted_labels = model.transduction_
confidence = np.max(model.label_distributions_, axis=1)
uncertain_count = int(np.sum(confidence < 0.8))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Manuell gelabelte Punkte: {len(known_indices)} von {len(features)}")
print(f"Weitergegebene Labels mit Sicherheit unter 0.80: {uncertain_count}")
print("Niedrige Sicherheit bedeutet: bitte später prüfen.")

# Die Punktgröße markiert manuell bekannte Labels.
point_sizes = np.full(len(features), 45)
point_sizes[known_indices] = 150

# Die Grafik zeigt Label-Weitergabe und Unsicherheit.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=predicted_labels,
    s=point_sizes,
    alpha=confidence,
    cmap="viridis",
)

ax.scatter(
    features[known_indices, 0],
    features[known_indices, 1],
    facecolors="none",
    edgecolors="black",
    s=220,
    linewidths=1.5,
    label="manuell gelabelt",
)

ax.set_title("Labels weitergeben: Farbe = Label, Transparenz = Sicherheit")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="best")
plt.show()



### **3.2. Labels sanft verbreiten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_02.jpg?v=1787649331" width="250">



>* Labels als Wahrscheinlichkeiten statt feste Entscheidungen
>* Ähnlichkeit zeigt Hinweise, aber keine Gewissheit

>* Ähnliche Daten teilen Label-Hinweise im Netzwerk
>* Weiche Zuordnung zeigt Cluster und Übergänge

>* Unsicherheit zeigt Grenzen weniger Labels
>* Gezielte Nachbeschriftung verbessert weitere Entscheidungen



In [ ]:
#@title Python-Code - Labels sanft verbreiten

# Wir verbreiten wenige Labels vorsichtig durch ähnliche Punkte.
# LabelSpreading zeigt Wahrscheinlichkeiten statt harter Entscheidungen.
# Unsichere Punkte werden sichtbar und gezielt prüfbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading

# Kleine künstliche Daten zeigen zwei überlappende Gruppen.
features, true_labels = make_blobs(
    n_samples=120, centers=2, cluster_std=1.35, random_state=42
)

# Skalierung macht Abstände für das Verfahren vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Nur vier Punkte behalten ihr bekanntes Label.
known_labels = np.full(true_labels.shape, -1)
known_indices = np.array([0, 1, 2, 3])
known_labels[known_indices] = true_labels[known_indices]

# LabelSpreading verteilt Hinweise weich über die Datengeometrie.
model = LabelSpreading(kernel="rbf", gamma=0.8, alpha=0.2, max_iter=100)
model.fit(scaled_features, known_labels)

# Wahrscheinlichkeiten beschreiben die Zuversicht jeder Zuordnung.
probabilities = model.label_distributions_
predicted_labels = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)

# Niedrige Zuversicht markiert gute Kandidaten für Nachbeschriftung.
uncertain_order = np.argsort(confidence)
uncertain_indices = uncertain_order[:5]
mean_confidence = confidence.mean()

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Bekannte Labels: {len(known_indices)} von {len(true_labels)} Punkten")
print(f"Mittlere Zuversicht nach sanfter Verbreitung: {mean_confidence:.2f}")
print("Fünf unsicherste Punkte: Index, Klasse, Zuversicht")

for point_index in uncertain_indices:
    label = predicted_labels[point_index]
    score = confidence[point_index]
    print(f"{point_index}: Klasse {label}, Zuversicht {score:.2f}")

# Die Farbe zeigt Klassen, die Größe zeigt Unsicherheit.
uncertainty = 1.0 - confidence
sizes = 40 + 260 * uncertainty
fig, ax = plt.subplots(figsize=(7, 5))

scatter = ax.scatter(
    scaled_features[:, 0], scaled_features[:, 1], c=predicted_labels,
    s=sizes, cmap="coolwarm", alpha=0.75, edgecolor="k"
)

# Gelbe Sterne markieren die wenigen wirklich bekannten Labels.
ax.scatter(
    scaled_features[known_indices, 0], scaled_features[known_indices, 1],
    marker="*", s=260, c="gold", edgecolor="black", label="bekannte Labels"
)

ax.set_title("Sanfte Label-Verbreitung mit sichtbarer Unsicherheit")
ax.set_xlabel("skaliertes Merkmal 1")
ax.set_ylabel("skaliertes Merkmal 2")
ax.legend(loc="best")
plt.show()



### **3.3. Unsicherheit dokumentieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_03.jpg?v=1787649330" width="250">



>* Automatische Labels unterschiedlich sicher bewerten
>* Unsichere Fälle gezielt prüfen lassen

>* Unsicherheit entsteht durch Modell, Daten und Abstand
>* Kennzeichnung unterstützt verantwortungsvolle Entscheidungen

>* Unsicherheit sichert Qualität und Nutzung.
>* Gezieltes Nachlabeln verbessert Modelle effizient.



In [ ]:
#@title Python-Code - Unsicherheit dokumentieren

# Wir dokumentieren Unsicherheit bei wenigen echten Labels.
# LabelSpreading liefert Wahrscheinlichkeiten für unbekannte Punkte.
# Unsichere Fälle werden sichtbar und prüfbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler

# Kleine synthetische Daten machen die Idee gut sichtbar.
features, true_labels = make_blobs(
    n_samples=90, centers=3, cluster_std=1.25, random_state=42
)

# Skalierung verhindert, dass eine Achse zu stark dominiert.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Nur wenige Punkte behalten ihr echtes Label.
known_indices = np.array([0, 1, 2, 30, 31, 32, 60, 61, 62])
partial_labels = np.full(true_labels.shape, -1)
partial_labels[known_indices] = true_labels[known_indices]

# Eine einfache Prüfung schützt vor einem unbrauchbaren Beispiel.
if len(np.unique(partial_labels[partial_labels != -1])) != 3:
    raise ValueError("Die wenigen Labels müssen alle drei Klassen enthalten.")

# LabelSpreading nutzt gelabelte und ungelabelte Punkte gemeinsam.
model = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=30)
model.fit(scaled_features, partial_labels)

# Hohe maximale Wahrscheinlichkeit bedeutet höhere Modell-Sicherheit.
predicted_labels = model.transduction_
confidence = model.label_distributions_.max(axis=1)
uncertainty = 1.0 - confidence

# Wir markieren Fälle, die menschliche Prüfung verdienen.
review_limit = 0.25
needs_review = uncertainty >= review_limit
review_count = int(needs_review.sum())

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Echte Labels: {len(known_indices)} von {len(partial_labels)} Punkten")
print(f"Zur Prüfung markiert: {review_count} Punkte")
print(f"Mittlere Unsicherheit: {uncertainty.mean():.2f}")

# Die Grafik zeigt sichere und unsichere automatische Labels.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0], scaled_features[:, 1], c=predicted_labels,
    s=45 + 220 * uncertainty, alpha=0.75, cmap="viridis"
)

# Schwarze Ringe zeigen die wenigen echten Labels.
ax.scatter(
    scaled_features[known_indices, 0], scaled_features[known_indices, 1],
    facecolors="none", edgecolors="black", s=180, linewidths=1.8,
    label="echtes Label"
)

# Rote Kreuze zeigen Fälle für menschliche Prüfung.
ax.scatter(
    scaled_features[needs_review, 0], scaled_features[needs_review, 1],
    marker="x", c="red", s=80, linewidths=2, label="prüfen"
)

ax.set_title("Unsicherheit bei automatisch vergebenen Labels")
ax.set_xlabel("skalierte Eigenschaft 1")
ax.set_ylabel("skalierte Eigenschaft 2")
ax.legend(loc="best")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Unüberwacht lernen**</font>


In this lecture, you learned to:
- Vergleichen KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture. 
- Wenden IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierte Daten an. 
- Untersuchen semi-supervised Verfahren mit wenigen Labels und dokumentieren Unsicherheit. 

In the next Lecture (Lecture B), we will go over 'Text als Merkmale'